# 03 · Exploratory analysis
Distributions, rankings, seasonality and time series of World Bank modelled PM2.5.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 30, "display.width", 160)

from src.data import load
from src.analysis import descriptive, trends, seasonality, correlations, statistical_tests as st, data_quality as dq, sensitivity
from src.visualization import charts, maps
from src.data.transform import independent_units

wb = descriptive.real_only(load.load_processed("wb_daily"))
cities = load.load_processed("wb_cities")

In [2]:
descriptive.dataset_overview(wb)

{'rows': 948183,
 'cities': 339,
 'date_min': Timestamp('2018-10-01 00:00:00'),
 'date_max': Timestamp('2026-08-30 00:00:00'),
 'n_dates': 2797,
 'n_values': 948183,
 'median': 59.3820080924779,
 'mean': 68.36482636107331,
 'missing_pct': 0.0}

In [3]:
descriptive.summary_stats(wb).T

,0
n,948183.000000
missing_pct,0.000000
mean,68.364826
std,41.674327
min,4.172080
p5,20.194570
p25,39.600550
p50,59.382008
p75,86.634368
p95,146.277818


In [4]:
charts.histogram(wb, log_x=True)

## City summary

In [5]:
cs = descriptive.city_summary(wb)
cs.head(15)

,city_id,city_label,province,latitude,longitude,population,n,mean,median,std,min,p95,max,pct_days_above_threshold,first_date,last_date,rank_by_median
0,wb:1172128,Larkana,Sindh,27.55898,68.21204,364033,2797,138.101864,126.116776,62.326501,30.904168,270.433728,354.681180,100.0,2018-10-01,2026-08-30,1
1,wb:1167460,Rahim Yar Khan,Punjab,28.41987,70.30345,517000,2797,124.046488,114.886493,54.924330,23.712232,231.595404,308.279256,100.0,2018-10-01,2026-08-30,2
2,wb:1169825,Multan,Punjab,30.19679,71.47824,2169915,2797,125.195498,111.446820,66.580908,16.996180,258.703878,377.034896,100.0,2018-10-01,2026-08-30,3
3,wb:1176889,Hingorja,Sindh,27.21088,68.41598,22263,2797,123.576003,111.307595,60.138717,25.620353,251.336155,327.742342,100.0,2018-10-01,2026-08-30,4
4,wb:1412008,Setharja Old,Sindh,27.21270,68.46883,32651,2797,123.576003,111.307595,60.138717,25.620353,251.336155,327.742342,100.0,2018-10-01,2026-08-30,4
5,wb:1179061,Gambat,Sindh,27.35170,68.52150,34005,2797,123.846228,110.584548,61.518419,24.640944,256.034458,336.995404,100.0,2018-10-01,2026-08-30,6
6,wb:1164408,Sukkur,Sindh,27.70323,68.85889,563851,2797,119.435096,110.097882,47.369443,27.690263,207.528797,264.435346,100.0,2018-10-01,2026-08-30,7
7,wb:1166827,Rohri,Sindh,27.69203,68.89503,50649,2797,119.435096,110.097882,47.369443,27.690263,207.528797,264.435346,100.0,2018-10-01,2026-08-30,7
8,wb:1169187,Naudero,Sindh,27.66684,68.36090,33455,2797,122.427799,109.083842,60.195372,25.150877,253.066593,340.107463,100.0,2018-10-01,2026-08-30,9
9,wb:1178456,Ghotki,Sindh,28.00437,69.31569,119879,2797,122.197201,108.508248,63.984890,23.297819,258.792839,356.521248,100.0,2018-10-01,2026-08-30,10


In [6]:
maps.city_metric_map(cs)

In [7]:
charts.province_box(cs)

## Seasonality

In [8]:
seasonality.seasonal_summary(wb)

,season,n,mean,median,p25,p75
0,Winter,236622,78.951190,68.405690,43.333709,101.761951
1,Spring,160008,63.577916,55.221779,37.009429,80.352747
2,Pre-monsoon,160008,64.841145,59.740053,41.622196,82.290641
3,Monsoon,231537,54.315289,49.819541,34.678211,68.497401
4,Post-monsoon,160008,81.350316,72.788482,46.846555,106.545413


In [9]:
charts.season_box(wb)

In [10]:
top = cs.sort_values('population', ascending=False)['city_label'].astype(str).head(20).tolist()
charts.month_city_heatmap(seasonality.month_city_matrix(wb[wb['city_label'].astype(str).isin(top)]).reindex(top))

In [11]:
charts.weekday_chart(seasonality.weekday_profile(wb))

## Time series

In [12]:
charts.national_band(trends.monthly_series(wb))

In [13]:
sel = top[:5]
d = wb[wb['city_label'].astype(str).isin(sel)]
charts.annual_lines(trends.annual_series(d, group_cols=('city_label',)))

## Hourly source (Kaggle/HF): pollutant distributions and diurnal profile
Hour labels are shown as given in the file (see notebook 02 for the timestamp diagnostic).

In [14]:
hf = load.load_processed('hf_hourly')
descriptive.summary_stats(hf, 'pm25', 'city').round(1)

,city,n,missing_pct,mean,std,min,p5,p25,p50,p75,p95,max,median
0,Faisalabad,2184.0,0.0,155.7,82.9,12.7,38.8,90.8,146.7,208.2,312.0,486.2,146.7
1,Islamabad,2184.0,0.0,56.1,28.1,11.9,20.5,34.3,49.1,74.1,110.1,146.2,49.1
2,Karachi,2184.0,0.0,37.7,15.0,8.1,17.0,26.7,35.8,46.1,67.1,95.9,35.8
3,Lahore,2184.0,0.0,126.8,63.1,16.8,39.5,78.0,121.1,166.2,239.3,353.9,121.1
4,Multan,2184.0,0.0,136.8,71.7,19.4,37.1,78.9,129.0,182.5,271.8,426.5,129.0
5,Peshawar,2184.0,0.0,53.7,30.0,6.0,16.1,29.2,48.7,72.4,110.6,156.5,48.7
6,Quetta,2184.0,0.0,19.1,11.0,0.5,4.6,10.3,17.6,26.1,38.6,68.2,17.6
7,Rahim Yar Khan,2184.0,0.0,72.4,34.8,10.9,24.9,47.1,68.5,90.8,137.7,199.3,68.5
8,Rawalpindi,2184.0,0.0,56.1,28.1,11.9,20.5,34.3,49.1,74.1,110.1,146.2,49.1
9,Sialkot,2184.0,0.0,115.2,59.3,15.5,39.0,68.9,104.5,150.9,225.7,364.1,104.5


In [15]:
charts.hour_profile_chart(seasonality.hour_profile(hf, 'pm25', by='city'))